# Chapter 8 &mdash; RE to NFA: the Thompson-Style Constructions

**Concept 2 of the Chapter 8 decomposition:** *RE to NFA: the Thompson-Style Constructions for $\varepsilon$, $a$, Concatenation, Union and Star*

One NFA fragment per RE operator, glued with $\varepsilon$ edges, with finality carefully managed.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter8/Concept-Thompson-Constructions/Concept-Thompson-Constructions.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_RE2NFA     import *
from jove.AnimateNFA     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


The RE-to-NFA conversion is **compositional**: one construction per operator, each
producing a fragment with **one start** and **one final** state so the next
construction can glue onto it.

* $\varepsilon$: a single state, initial and final.
* $a$: two states with an $a$-edge.
* $R_1R_2$: $\varepsilon$ from $R_1$'s final to $R_2$'s start; $R_1$'s final stops being final.
* $R_1+R_2$: a new start with $\varepsilon$ edges to both.
* $R^*$: a new start-and-final state, $\varepsilon$ into $R$, and $\varepsilon$ back from
  $R$'s final.

The $\varepsilon$ edges are what keep each rule to two lines &mdash; this is Concept 3 of
Chapter 7 paying off.

## 2. Definitions

### Jove's fragment builders, used directly

In [ ]:
from jove.Def_RE2NFA import (mk_eps_nfa, mk_symbol_nfa, mk_cat_nfa,
                             mk_plus_nfa, mk_star_nfa, ResetStNum)
def sizes(N): return "|Q|=%d Q0=%s F=%s" % (len(N["Q"]), sorted(N["Q0"]), sorted(N["F"]))

### The same thing via the parser, for comparison

In [ ]:
def re_dfa(r): return min_dfa(nfa2dfa(re2nfa(r)))

## 3. Tests

The two primitives.

In [ ]:
ResetStNum()
E = mk_eps_nfa()
A = mk_symbol_nfa('0')
print("epsilon fragment :", sizes(E))
print("symbol  fragment :", sizes(A))
assert accepts_nfa(E, '') and accepts_nfa(A, '0')
assert E["Q0"] & E["F"], "for epsilon the start state is also final"

**Concatenation** glues with one $\varepsilon$ edge and demotes the left fragment's final state.

In [ ]:
ResetStNum()
cat = mk_cat_nfa(mk_symbol_nfa('0'), mk_symbol_nfa('1'))
print("cat fragment :", sizes(cat))
assert accepts_nfa(cat, '01')
assert not accepts_nfa(cat, '0') and not accepts_nfa(cat, '1')
print("accepts '01' only -- the left fragment's old final state is no longer final")

**Union** adds a new start with two $\varepsilon$ edges.

In [ ]:
ResetStNum()
alt = mk_plus_nfa(mk_symbol_nfa('0'), mk_symbol_nfa('1'))
print("union fragment :", sizes(alt))
assert accepts_nfa(alt, '0') and accepts_nfa(alt, '1')
assert not accepts_nfa(alt, '01')

**Star** adds a start-and-final state with $\varepsilon$ in and $\varepsilon$ back.

In [ ]:
ResetStNum()
st = mk_star_nfa(mk_symbol_nfa('0'))
print("star fragment :", sizes(st))
for s in ['', '0', '00', '000']:
    assert accepts_nfa(st, s), s
print("accepts epsilon and every 0^n -- the new state is initial AND final")

The parser composes the same fragments; the languages agree.

In [ ]:
built  = mk_star_nfa(mk_plus_nfa(mk_symbol_nfa('0'), mk_symbol_nfa('1')))
parsed = re2nfa("(0+1)*")
print("iso after minimizing? ",
      iso_dfa(min_dfa(nfa2dfa(built)), min_dfa(nfa2dfa(parsed))))
assert iso_dfa(min_dfa(nfa2dfa(built)), min_dfa(nfa2dfa(parsed)))

Fragment sizes stay **linear** in the RE length &mdash; that is the point of the construction.

In [ ]:
for r in ["0", "01", "0+1", "(0+1)*", "(0+1)*1(0+1)(0+1)"]:
    print("%-20s RE length %2d -> NFA |Q| = %2d" % (r, len(r), len(re2nfa(r)["Q"])))

## 4. Animation

A Thompson-built NFA: notice the $\varepsilon$ edges doing the gluing.

*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimateNFA import *
AnimateNFA(re2nfa('(0+1)*1'), FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Draw the star construction by hand. Why does the new state have to be final?
2. What goes wrong if concatenation forgets to demote the left fragment's final state?
3. How many states does the Thompson NFA for an RE of length $n$ have, roughly?

In [ ]:
# Your work for the exercises above.